# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Clasificación de relaciones con raruidol/ArgumentMining-EN-ARI-AIF-RoBERTa_L

Model page: https://huggingface.co/raruidol/ArgumentMining-EN-ARI-AIF-RoBERTa_L

# No keywords

In [1]:
import os
import re
import math
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gc

process_rel_path = r"/kaggle/working/TFM/Data/Relationships No Keywords"
hf_model_repo = "raruidol/ArgumentMining-EN-ARI-AIF-RoBERTa_L"
model_name = "robertaL"  

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(hf_model_repo)
model = AutoModelForSequenceClassification.from_pretrained(hf_model_repo).to(device)
model.eval()


tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/945 [00:00<?, ?B/s]

2025-08-24 11:27:40.187663: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756034860.508187      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756034860.598141      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=Tru

In [2]:
!git clone https://github.com/camipalo/TFM.git

Cloning into 'TFM'...
remote: Enumerating objects: 2147, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 2147 (delta 176), reused 162 (delta 112), pack-reused 1920 (from 1)
Receiving objects: 100% (2147/2147), 69.26 MiB | 14.79 MiB/s, done.
Resolving deltas: 100% (1815/1815), done.
Updating files: 100% (1008/1008), done.


In [3]:
# --- compute max token length across all input pairs ---
def compute_max_length(input_dir, prefix_substring, tokenizer, safety_limit=512):
    max_len = 0
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    for fn in files:
        path = os.path.join(input_dir, fn)
        df = pd.read_csv(path)
        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            continue
        for a, b in zip(df["SDGarg1"], df["SDGarg2"]):
            a = str(a) if pd.notna(a) else ""
            b = str(b) if pd.notna(b) else ""
            tokens = tokenizer(a, b, truncation=False, padding=False)["input_ids"]
            max_len = max(max_len, len(tokens))
    return min(max_len, safety_limit)

### Labels
id2label = getattr(model.config, "id2label", None) or {0: "Inference", 1: "Conflict", 2: "Rephrase"}
id2label = {int(k): str(v) for k, v in id2label.items()}

# Map model labels -> desired labels
LABEL_MAP = {
    "inference": "Support",
    "conflict": "Attack",
    "rephrase": "Rephrase",
    "none": "No Relationship",
    "0": "Support",
    "1": "Attack",
    "2": "Rephrase",
    "-1": "No Relationship",
}

VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}

def preprocess_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

@torch.no_grad()
def predict_labels(pairs):
    if not pairs:
        return []
    enc = tokenizer(
        [preprocess_text(a) for a, _ in pairs],
        [preprocess_text(b) for _, b in pairs],
        truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt"
    ).to(device)
    logits = model(**enc).logits  # [batch, num_labels]
    preds = torch.argmax(logits, dim=-1).tolist()
    raw = [id2label.get(int(p), "none") for p in preds]
    mapped = []
    for r in raw:
        key = str(r).strip().lower()
        mapped.append(LABEL_MAP.get(key, "No Relationship"))
    return mapped

def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
def classify_relationships_hf(input_dir: str, prefix_substring: str):
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    if not files:
        print(f"No CSVs found in '{input_dir}' containing '{prefix_substring}'.")
        return

    for fn in files:
        path = os.path.join(input_dir, fn)
        print(f"\nProcessing: {path}")
        df = pd.read_csv(path)

        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            print(f"  Skipped (missing SDGarg1/SDGarg2): {fn}")
            continue

        # init column
        df[rel_col] = ""

        n = len(df)
        total_done = 0
        first_examples = []

        batches = math.ceil(n / BATCH_SIZE)
        for bi in range(batches):
            s = bi * BATCH_SIZE
            e = min((bi + 1) * BATCH_SIZE, n)
            chunk = df.iloc[s:e]
            pairs = list(zip(chunk["SDGarg1"].astype(str).tolist(),
                             chunk["SDGarg2"].astype(str).tolist()))
            try:
                labels = predict_labels(pairs)
            except Exception as ex:
                print(f"  Batch {bi+1}/{batches} error: {ex}. Marking 'No Relationship'.")
                labels = ["No Relationship"] * len(pairs)

            df.loc[chunk.index, rel_col] = labels

            # collect first 5 examples
            for (a1, a2), lab in zip(pairs, labels):
                if len(first_examples) < 5:
                    first_examples.append((a1, a2, lab))
                elif len(first_examples) == 5:
                    print("Check first 5 predictions labels:\n")
                    for a1, a2, lab in first_examples:
                        print(f"- Arg1: {a1[:]} \n-Arg2: {a2[:]} \nLabel: {lab}\n\n")
                    first_examples.append((a1, a2, lab))

            total_done += len(pairs)
            if total_done % 50 < BATCH_SIZE:  
                print(f"  Progress: {total_done}/{n} relations classified...")

        # handle missing args
        mask_nan = df["SDGarg1"].isna() | df["SDGarg2"].isna()
        df.loc[mask_nan, rel_col] = "No Relationship"

        # Ensure valid set
        bad = ~df[rel_col].isin(VALID_OUT)
        if bad.any():
            df.loc[bad, rel_col] = "No Relationship"

        df.to_csv(path, index=False, encoding="utf-8")
        free_cuda()
        print(f"Saved: {path}")
        return df

## GLOBAL SDG 2023 

#### Qwen2.5 3B extraction

In [4]:
prefix = "intra_goalGLOBAL_SGD2023_qwen2.5-3b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 171

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Check first 5 predictions labels:

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
-Arg2: According to major international studies, few teenagers can differentiate between a fact and an opinion. 
Label: No Relationship


- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
-Arg2: As the world’s nations prepare to meet in September to review the progress the world has made so far towards achieving the SDGs, at the midpoint of the 2030 Agenda, SDSN emphasizes six areas for immediate action. 
Label: Support


- Arg1: At their core, the SDGs are an investment 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_robertaL
639,Zero-carbon energy systems: the transition by ...,Use of unilateral coercive measures,7_0,7_9,NaN,No Relationship,No Relationship
910,Grave dangers of social tipping points,"The GFA includes multilateral institutions, na...",16_0,16_3,NaN,No Relationship,No Relationship
142,The SDGs are not only a public policy framewor...,Rich European countries top the overall SDG In...,0_4,0_29,NaN,No Relationship,Support
736,And global cooperation has ebbed as geopolitic...,Governments are only now learning how to desig...,10_1,10_8,NaN,No Relationship,No Relationship
976,"UN Member States should adopt an SDG Stimulus,...",SDG 17 (Partnerships for the Goals) calls for ...,17_1,17_11,NaN,Support,Rephrase


rel_robertaL
Support            616
No Relationship    378
Rephrase            47
Attack              17
Name: count, dtype: int64

In [5]:
prefix = "cross_goalGLOBAL_SGD2023_qwen2.5-3b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 181

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Check first 5 predictions labels:

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
-Arg2: only limited progress is being made on the environmental and biodiversity goals, including SDG 12 (Responsible Consumption and Production), SDG 13 (Climate Action), SDG 14 (Life Below Water), and SDG 15 (Life on Land) 
Label: Support


- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
-Arg2: And global cooperation has ebbed as geopolitical tensions have risen. 
Label: No Relationship


- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_robertaL
434,Investing in the SDGs is an investment agenda....,The SDG Index acknowledges Bhutan’s recent pro...,0_6,3_10,NaN,Support,Support
2782,47 countries have submitted a VNR this year: o...,Only limited progress is being made on the env...,0_19,12_0,NaN,No Relationship,No Relationship
3869,"For these reasons, we end our message with two...",SDG 17 (Partnerships for the Goals) calls for ...,0_14,17_11,NaN,No Relationship,No Relationship
4637,"12. Sachs JD, Schmidt-Traub G., 2020. Speaking...",the COVID-19 pandemic has had lasting impacts ...,1_3,10_11,NaN,No Relationship,No Relationship
10485,HICs tend to generate the largest negative spi...,"UN Member States should adopt an SDG Stimulus,...",10_9,17_1,NaN,Support,No Relationship


rel_robertaL
Support            5670
No Relationship    5516
Rephrase            452
Attack              184
Name: count, dtype: int64

In [6]:
pd.crosstab(
    args_classified["rel_llama"],
    args_classified["rel_robertaL"],
    margins=True,        # adds totals
    margins_name="Total" # name for totals row/col
)

rel_robertaL,Attack,No Relationship,Rephrase,Support,Total
rel_llama,,,,,
Attack,96,600,51,795,1542
No Relationship,34,3934,91,1246,5305
Rephrase,0,6,118,13,137
Support,54,976,192,3616,4838
Total,184,5516,452,5670,11822


#### Gemma3 27B extraction

In [7]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 213

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_gemma3-27b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Despite this alarming development, the SDGs are still achievable. 
Label: Attack


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: National governments mus

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_robertaL
651,"Despite significant efforts in some places, na...",Achieving the SDGs will require a transformati...,0_14,0_29,NaN,Support,Support
1386,"At the global level, averaging across countrie...",The poor are consequently languishing in poverty.,1_0,1_9,NaN,Support,Support
2464,"Global warming as of 2022 stood at 1.2°C, with...",Human-induced global warming could hit several...,13_2,13_8,NaN,Support,Support
3386,These fora are critical to encourage internati...,"There are no magic numbers, but rather a suite...",16_19,16_26,NaN,Support,Support
3746,And global cooperation has ebbed as geopolitic...,We therefore consider that promoting multilate...,17_9,17_33,NaN,Support,Support


rel_robertaL
Support            3117
No Relationship     710
Rephrase            248
Attack              113
Name: count, dtype: int64

In [8]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 262

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_gemma3-27b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: As called for by United Nations Secretary-General António Guterres, the SDG Stimulus plan has five main components: 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Greatly increased funding for national and subnational governments and private businesses in the emerging economies, especially the low-income countries (LICs) and lower-middle-income countries (LMICs), to carry out needed SDG actions; 
Label: Suppor

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_robertaL
23186,"In the SDGs, universal health care (UHC) is co...",Countries must further expand and transform ed...,3_12,4_9,NaN,No Relationship,Support
47720,The SDG Index and Dashboards also include unof...,"Second, developed countries are not being held...",14_6,17_16,NaN,Attack,Support
17541,Greatly increased funding for national and sub...,More ambitious policies and actions on climate...,1_2,13_26,NaN,Support,Rephrase
13732,Achieving the SDGs will require a transformati...,We firmly believe that international cooperati...,0_29,17_11,NaN,Support,Support
31997,Although on average the world has made some pr...,Questions explored policy measures to address ...,6_6,13_25,NaN,Support,Rephrase


rel_robertaL
Support            25808
No Relationship    22354
Rephrase            1338
Attack               597
Name: count, dtype: int64

#### Gemma3 4B extraction

In [9]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-4b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 270

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_gemma3-4b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: None of their objectives are beyond our reach. 
Label: Attack


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The SDGs are still achievable. 
Label: Attack


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: It is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, al

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL
8617,Long-term investment plans are essential for n...,The IMF should build its national reviews (Art...,0_64,0_74,NaN,Support
9539,The World Bank and the other MDBs should put t...,no single G20 country has adopted a sufficient...,0_73,0_123,NaN,No Relationship
22025,All UN Member States and UN agencies can count...,It is widely recognized that the world needs t...,17_4,17_19,NaN,No Relationship
8778,SDSN has recommended six inter-related long-te...,UN Specialized Agencies: the Food and Agricult...,0_65,0_134,NaN,Support
4604,includes equality of opportunities for girls a...,Governments are only now establishing R&D fund...,0_30,0_90,NaN,No Relationship


rel_robertaL
Support            12040
No Relationship     9586
Rephrase             874
Attack               414
Name: count, dtype: int64

In [10]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-4b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 270

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_gemma3-4b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The world is off track, but that is all the more reason to double down on the SDGs. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: It is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Revise liquidity structures for LICs and LMICs, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises. 
Label: No Relationship


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Create ambitious, i

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL
113650,not a single SDG is projected to be met at the...,The HICs’ somewhat better performance on pilla...,2_1,3_33,NaN,No Relationship
192637,Revision of the liquidity structures for LICs ...,Many countries continue to provide substantial...,9_6,13_22,NaN,No Relationship
201267,"LICs, LMICs, and SIDS are highly vulnerable to...",This is driven in part by the moderate or low ...,10_22,13_30,NaN,Support
215380,Food systems account for a quarter of greenhou...,It is also vital to share fairly and globally ...,12_21,17_24,NaN,Support
186746,"At the midpoint of the 2030 Agenda, all countr...",Governments are only now establishing R&D fund...,8_19,16_29,NaN,No Relationship


rel_robertaL
No Relationship    116729
Support             98293
Rephrase             7259
Attack               3670
Name: count, dtype: int64

#### Llama3.3 70B extraction

In [11]:
prefix = "intra_goalGLOBAL_SGD2023_llama3.3-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 247

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_llama3.3-70b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Since the outbreak of the pandemic in 2020 and other simultaneous crises, SDG progress has stalled globally. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The world is off track, but that is all the more reason to double down on the SDGs. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: To achieve the SDGs

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_robertaL
7481,According to the UNEP Emissions Gap Report 202...,governments to combine the objectives of econo...,13_5,13_29,NaN,No Relationship,No Relationship
7088,5. Sustainable cities,SDSN recognizes this gap in the practical mean...,11_3,11_7,NaN,Support,Support
9612,We commend global leaders who “oppose the use ...,The SDG Index is prepared by an independent gr...,16_21,16_37,NaN,No Relationship,No Relationship
10396,"Moreover, societal polarization, populism, and...",Both sides – importers and exporters – must wo...,17_10,17_49,NaN,Support,Support
10905,"Second, developed countries are not being held...",All UN Member States and United Nations agenci...,17_20,17_43,NaN,No Relationship,Attack


rel_robertaL
Support            8625
No Relationship    2465
Rephrase            635
Attack              286
Name: count, dtype: int64

In [12]:
prefix = "cross_goalGLOBAL_SGD2023_llama3.3-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 296

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_llama3.3-70b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Similarly, extreme poverty can lead to a collapse of tax revenues, followed by government bankruptcy and further economic collapse, a syndrome that now threatens dozens of poor countries. 
Label: No Relationship


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL
9373,Non-traditional statistics and science-based p...,Although on average the world has made some pr...,0_108,6_7,NaN,No Relationship
36667,"At the mid-point of the SDG agenda, we are far...",International cooperation is trapped by bureau...,0_58,17_23,NaN,Attack
24217,Since the outbreak of the pandemic in 2020 and...,incorporating recent international agreements ...,0_1,14_7,NaN,No Relationship
62522,The International Commission on the Futures of...,"Working together with the IMF and the MDBs, th...",4_12,17_30,NaN,No Relationship
72630,National government must also work with subnat...,Many urban organizations and associations have...,9_7,11_10,NaN,Support


rel_robertaL
No Relationship    53172
Support            44980
Rephrase            2509
Attack               854
Name: count, dtype: int64

#### Deepseek r1 70B extraction

In [4]:
prefix = "intra_goalGLOBAL_SGD2023_deepseek-r1-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 372

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: Despite this alarming development, the SDGs are still achievable. None of their objectives are beyond our reach. 
Label: Attack


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The world is off track, but that is all the more reason to double down on the SDGs. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: To achieve the

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL
4855,"At their core, the SDGs are an investment agenda.","Finally, the SDG Index contributes to global e...",0_39,0_137,NaN,Support
6107,Some other G20 countries have shown weak commi...,Global infrastructure programs like China’s Be...,0_53,0_66,NaN,Support
6431,Many of them lack an adequately high SDG commi...,The SDG Index is a flagship instrument to prom...,0_56,0_132,NaN,Support
11716,1. Universal quality education and innovation-...,5. Private capital markets continue to direct ...,9_0,9_4,NaN,No Relationship
6378,Many of them lack an adequately high SDG commi...,The SDG Academy is also building partnerships ...,0_56,0_79,NaN,Support


rel_robertaL
Support            11430
No Relationship     4690
Rephrase             676
Attack               317
Name: count, dtype: int64

In [5]:
prefix = "cross_goalGLOBAL_SGD2023_deepseek-r1-70b"   
rel_col = f"rel_{model_name}"
BATCH_SIZE = 250
MAX_LENGTH = compute_max_length(process_rel_path, prefix, tokenizer)
print(f"Using MAX_LENGTH = {MAX_LENGTH}")

args_classified = classify_relationships_hf(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_robertaL"].value_counts()

Using MAX_LENGTH = 398

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Check first 5 predictions labels:

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments. 
Label: Support


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most. 
Label: Rephrase


- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track. 
-Arg2: 1. Increased funding from the mult

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL
55874,"According to the OECD, only one in 10 students...",Open sharing of data and knowledge across thes...,0_109,17_53,NaN,Support
89948,"The SDSN’s flagship educational initiative, th...",National governments must ensure both the dome...,4_8,17_20,NaN,No Relationship
122524,We also need responsible business leadership l...,Current geopolitical tensions are hindering SD...,12_5,17_15,NaN,Support
60587,"Education builds human capital, which in turn ...",An estimated 1.8 billion people depend on drin...,1_22,6_1,NaN,No Relationship
29783,"The collaboration with SDSN is ongoing, with t...",More than half of the 192 local and regional g...,0_139,11_14,NaN,Support


rel_robertaL
No Relationship    70902
Support            59883
Rephrase            3156
Attack              1574
Name: count, dtype: int64